# LOGO Retraining — Leave-One-Generator-Out (cross-generator generalization)

For each held-out generator this notebook trains MFFT-Base from scratch on
`dataset/metadata/logo/logo_train_wo_<gen>.csv` (all real images + every OTHER
generator's fakes) and evaluates on `logo_test_<gen>.csv` (the held-out
generator's fakes + unseen real images). The mean accuracy across held-out
generators is the paper's headline generalization result (RQ4 / H3).

- Manifests are generated automatically if missing (same as `paper_evals.ipynb` Cell 10).
- No GPU -> SMOKE mode: 1 generator, tiny subset, 1 epoch — verifies the code path.
- GPU -> all generators, `LOGO_EPOCHS` (default 10; bump to 20 if allocation allows).

In [1]:
# Cell 1: Imports & Setup
import os, sys, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.notebook import tqdm

_p = Path.cwd().resolve()
for __ in range(10):
    if (_p / 'AGENTS.md').exists() or (_p / '.git').exists():
        break
    _parent = _p.parent
    if _parent == _p:
        break
    _p = _parent
PROJECT_ROOT = _p
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'model'))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SMOKE_TEST = not torch.cuda.is_available()
print(f'Root: {PROJECT_ROOT} | Device: {device} | SMOKE_TEST: {SMOKE_TEST}')

Root: G:\ai-image-detection-research | Device: cpu | SMOKE_TEST: True


In [2]:
# Cell 2: Config + ensure LOGO manifests exist
from src.dataset import create_split_dataloaders
from src.model import build_mfft, count_parameters
from src.logo_eval import make_logo_manifests, evaluate_per_generator, FAKE_GENERATORS

VARIANT = 'base'
LOGO_EPOCHS = 1 if SMOKE_TEST else 10   # 10 is an acceptable LOGO budget; 20 if time allows
IMAGE_SIZE = 224 if SMOKE_TEST else 384
BATCH = 8 if SMOKE_TEST else 64
WORKERS = 0 if SMOKE_TEST else 8
MAX_SAMPLES = 600 if SMOKE_TEST else None
EVAL_PER_GEN = 40 if SMOKE_TEST else 5000

_manifest = PROJECT_ROOT / 'dataset' / 'metadata' / 'train_manifest.csv'
if not _manifest.exists():
    _manifest = PROJECT_ROOT / 'dataset' / 'metadata' / 'clean_metadata.csv'
    print('WARNING: using clean_metadata.csv (run prepare_training_manifest.py for full scale)')
MANIFEST = str(_manifest)

GENERATORS = FAKE_GENERATORS[:1] if SMOKE_TEST else FAKE_GENERATORS
logo_dir = PROJECT_ROOT / 'dataset' / 'metadata' / 'logo'
manifests = {}
for g in GENERATORS:
    slug = g.replace(' ', '_').replace('-', '_').lower()
    paths = {'train': str(logo_dir / f'logo_train_wo_{slug}.csv'),
             'test': str(logo_dir / f'logo_test_{slug}.csv')}
    if not (Path(paths['train']).exists() and Path(paths['test']).exists()):
        paths = make_logo_manifests(MANIFEST, g, out_dir=str(logo_dir), seed=SEED)
    manifests[g] = paths

OUT = PROJECT_ROOT / 'paper' / 'result' / ('verify' if SMOKE_TEST else 'full_scale') / 'logo'
OUT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT = PROJECT_ROOT / 'model' / 'checkpoints' / ('verify' if SMOKE_TEST else '') / 'logo'
print(f'Generators: {GENERATORS}')
print(f'Epochs per run: {LOGO_EPOCHS} | Outputs -> {OUT}')

Generators: ['BigGAN']
Epochs per run: 1 | Outputs -> G:\ai-image-detection-research\paper\result\verify\logo


In [3]:
# Cell 3: Train + evaluate per held-out generator
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR

AMP = torch.cuda.is_available()
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
summary_rows = []

for gen, paths in manifests.items():
    slug = gen.replace(' ', '_').replace('-', '_').lower()
    print('\n' + '=' * 70)
    print(f'LOGO: holding out {gen}')
    print('=' * 70)

    # train/val from the LOGO train manifest (its own split file; the TEST
    # set is fully defined by the logo_test manifest, so the small test
    # portion of this split is unused)
    train_loader, val_loader, _ = create_split_dataloaders(
        root_dir=str(PROJECT_ROOT), metadata_paths=[paths['train']],
        batch_size=BATCH, num_workers=WORKERS, size=IMAGE_SIZE,
        val_split=0.05, test_split=0.05, seed=SEED, use_weighted_sampler=True,
        split_index_path=str(logo_dir / f'split_{slug}{"_smoke" if SMOKE_TEST else ""}.json'),
        max_samples=MAX_SAMPLES,
    )

    model = build_mfft(VARIANT).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0,
                      total_iters=min(500, len(train_loader)))
    cosine = CosineAnnealingWarmRestarts(
        optimizer, T_0=max(1, LOGO_EPOCHS * len(train_loader)), T_mult=2, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                             milestones=[min(500, len(train_loader))])
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)

    best_val, ckpt_dir = 0.0, CKPT_ROOT / slug
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    for epoch in range(LOGO_EPOCHS):
        model.train()
        correct = total = 0
        for images, labels in tqdm(train_loader, desc=f'{slug} E{epoch+1}/{LOGO_EPOCHS}'):
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda', enabled=AMP):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            correct += (logits.argmax(-1) == labels).sum().item()
            total += labels.size(0)

        model.eval()
        v_correct = v_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                v_correct += (model(images).argmax(-1) == labels).sum().item()
                v_total += labels.size(0)
        val_acc = v_correct / max(v_total, 1) * 100
        print(f'  epoch {epoch+1}: train={correct/max(total,1)*100:.2f}% val={val_acc:.2f}%')
        if val_acc >= best_val:
            best_val = val_acc
            torch.save(model.state_dict(), ckpt_dir / 'best.pt')

    # evaluate on the held-out generator's test manifest
    model.load_state_dict(torch.load(ckpt_dir / 'best.pt', map_location=device))
    rows = evaluate_per_generator(
        model, paths['test'], images_root=str(PROJECT_ROOT / 'dataset' / 'images'),
        device=str(device), size=IMAGE_SIZE, batch_size=BATCH,
        max_per_generator=EVAL_PER_GEN,
        out_csv=str(OUT / f'logo_{slug}_detail.csv'),
    )
    by_group = {r['generator']: r for r in rows}
    held_acc = by_group.get(gen, {}).get('accuracy', float('nan'))
    real_acc = by_group.get('real', {}).get('accuracy', float('nan'))
    auc = by_group.get('OVERALL_AUC', {}).get('accuracy', float('nan'))
    summary_rows.append({
        'held_out': gen, 'epochs': LOGO_EPOCHS, 'best_val_acc': round(best_val, 2),
        'heldout_fake_acc': held_acc, 'unseen_real_acc': real_acc,
        'balanced_acc': round((float(held_acc) + float(real_acc)) / 2, 2),
        'auc': auc,
    })
    print(f'==> {gen}: held-out fake acc={held_acc}%, real acc={real_acc}%, AUC={auc}')


LOGO: holding out BigGAN


Dataset loaded: 2176407 samples
  Real: 1767204, AI: 409203, Total: 2176407


max_samples: reduced to 600 balanced samples


Saved split indices to G:\ai-image-detection-research\dataset\metadata\logo\split_biggan_smoke.json
Dataset loaded: 0 samples
  Real: 0, AI: 0, Total: 0
Dataset loaded: 0 samples
  Real: 0, AI: 0, Total: 0
Dataset loaded: 0 samples
  Real: 0, AI: 0, Total: 0

Train: 540  Val: 30  Test: 30


biggan E1/1:   0%|          | 0/67 [00:00<?, ?it/s]

G:\ai-image-detection-research\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  epoch 1: train=63.99% val=50.00%


              BigGAN: acc=0.00%  (n=40)


                real: acc=100.00%  (n=40)
         overall AUC: 0.8738
Saved per-generator table to G:\ai-image-detection-research\paper\result\verify\logo\logo_biggan_detail.csv
==> BigGAN: held-out fake acc=0.0%, real acc=100.0%, AUC=0.8738


In [4]:
# Cell 4: LOGO summary table (the paper's cross-generator headline)
df = pd.DataFrame(summary_rows)
df.to_csv(OUT / 'logo_summary.csv', index=False)
print(df.to_string(index=False))
if len(df):
    print(f"\nMean held-out balanced accuracy: {df['balanced_acc'].mean():.2f}%")
print(f'\nSaved {OUT / "logo_summary.csv"}')
print('LOGO COMPLETE - no errors')

held_out  epochs  best_val_acc  heldout_fake_acc  unseen_real_acc  balanced_acc    auc
  BigGAN       1          50.0               0.0            100.0          50.0 0.8738

Mean held-out balanced accuracy: 50.00%

Saved G:\ai-image-detection-research\paper\result\verify\logo\logo_summary.csv
LOGO COMPLETE - no errors
